# 04B - SIMCA Optuna Search

This notebook runs Optuna challenger searches for the new workflow. The search space uses the same PCA preprocessing scope as 04A, with object and pixel matrix families kept separate.

It does not select final models. It writes all completed Optuna model candidates and a separate table containing only candidates not already present in the 04A grid.

## Inputs and outputs

Inputs:
- `results/03_pca_<RESULTS_TAG>/pca_selected_preprocessings.parquet`
- `results/04A_simca_grid_search_<RESULTS_TAG>/grid_model_candidates.parquet`
- `HSI Data/processed/nir_uco_database.h5`

Outputs:
- `optuna_trials.parquet`: all Optuna trials
- `optuna_completed_trials.parquet`: completed trials only
- `optuna_model_candidates.parquet`: all deduplicated Optuna candidates
- `optuna_new_model_candidates.parquet`: candidates not already generated by 04A
- `optuna_candidate_metrics.parquet`: validation 2-way metrics for Optuna candidates
- `optuna_search_protocol.parquet`: run configuration

In [1]:
from pathlib import Path
import sys
import json

import numpy as np
import optuna
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError(
        "Could not find project root. Run this notebook from the project root or notebooks/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

from src import experiment_config as expcfg
from src.io.database_h5 import load_nir_uco_h5
from src.utils import load_parquet, save_parquet, list_result_files
from src.workflows.simca_tables import compact_simca_table_for_path
from src.spectra.band_selection import (
    select_wavelength_range_from_database,
    wavelength_selection_summary,
)
from src.workflows.simca import make_target_train_filters
from src.workflows.simca_candidates import (
    add_selection_track,
    build_pca_preprocessing_configs_by_matrix_family,
    deduplicate_simca_candidates,
    filter_simca_candidates_by_pca_preprocessing,
    validate_simca_candidate_contract,
    validate_simca_evaluation_contract,
)
from src.workflows.simca_optuna import (
    close_optuna_study,
    make_optuna_binary_pareto_objective,
    optuna_trials_dataframe,
    optuna_trials_to_candidate_configs,
)

PROJECT_ROOT: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts


## Configuration

Optuna is run separately for object and pixel matrix families. The saved candidates use the same candidate contract as 04A so that 04C can concatenate both sources directly.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DB_H5_PATH = PROJECT_ROOT / "HSI Data" / "processed" / "nir_uco_database.h5"

WAVELENGTH_MODE = expcfg.DEFAULT_WAVELENGTH_MODE
RESULTS_TAG = expcfg.DEFAULT_RESULTS_TAG
RESULTS_03_DIR = PROJECT_ROOT / "results" / f"03_pca_{RESULTS_TAG}"
RESULTS_04A_DIR = PROJECT_ROOT / "results" / f"04A_simca_grid_search_{RESULTS_TAG}"
RESULTS_DIR = PROJECT_ROOT / "results" / f"04B_simca_optuna_search_{RESULTS_TAG}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PCA_SELECTED_PREPROCESSINGS_PATH = RESULTS_03_DIR / "pca_selected_preprocessings.parquet"
GRID_MODEL_CANDIDATES_PATH = RESULTS_04A_DIR / "grid_model_candidates.parquet"

OPTUNA_TRIALS_PATH = RESULTS_DIR / "optuna_trials.parquet"
OPTUNA_COMPLETED_TRIALS_PATH = RESULTS_DIR / "optuna_completed_trials.parquet"
OPTUNA_MODEL_CANDIDATES_PATH = RESULTS_DIR / "optuna_model_candidates.parquet"
OPTUNA_NEW_MODEL_CANDIDATES_PATH = RESULTS_DIR / "optuna_new_model_candidates.parquet"
OPTUNA_CANDIDATE_METRICS_PATH = RESULTS_DIR / "optuna_candidate_metrics.parquet"
OPTUNA_DIAGNOSTICS_PATH = RESULTS_DIR / "optuna_search_diagnostics.parquet"
OPTUNA_PROTOCOL_PATH = RESULTS_DIR / "optuna_search_protocol.parquet"
WAVELENGTH_CONFIG_PATH = RESULTS_DIR / "wavelength_config.parquet"
PREPROCESSING_SCOPE_PATH = RESULTS_DIR / "preprocessing_scope.parquet"

TARGET_CLASS = expcfg.TARGET_CLASS
NON_TARGET_LABEL = expcfg.NON_TARGET_LABEL
REFERENCE_CLASSES = list(expcfg.REFERENCE_CLASSES)

TRAIN_FILTERS = make_target_train_filters(
    target_class=TARGET_CLASS,
    train_batches=expcfg.SIMCA_TRAIN_BATCHES,
)
VALIDATION_FILTERS = {
    "sample_kind": ["pure"],
    "object_nut_type": REFERENCE_CLASSES,
    "batch": list(expcfg.SIMCA_VALIDATION_BATCHES),
}

OPTUNA_MATRIX_FAMILY_SPACES = {
    "object_matrix": {
        "matrix_methods": ["object_mean", "object_median"],
        "study_name": f"04B_object_matrix_{RESULTS_TAG}",
        "storage_path": RESULTS_DIR / "optuna_object_matrix.sqlite3",
    },
    "pixel_matrix": {
        "matrix_methods": ["balanced_pixels"],
        "study_name": f"04B_pixel_matrix_{RESULTS_TAG}",
        "storage_path": RESULTS_DIR / "optuna_pixel_matrix.sqlite3",
    },
}

RULE_VARIANTS = [
    "simple_chi2",
    "simple_emp_cv",
    "alternative_chi2_fixed2",
    "alternative_chi2_emp_cv",
    "alternative_empHQ_fixed2",
    "alternative_empHQ_emp_cv",
    "data_driven_chi2",
    "data_driven_emp_cv",
    "combined_index_chi2",
]

N_COMPONENTS_CHOICES = [3, 4, 5, 6, 7, 8, 10, 11, 12]
ALPHA_CHOICES = list(expcfg.SIMCA_ALPHA_VALUES)
OBJECT_THRESHOLDS = list(expcfg.SIMCA_OBJECT_THRESHOLDS)
M_CHOICES = [20, 40, 60, 80]
BALANCED_PIXEL_STRATEGY_CHOICES = list(expcfg.BALANCED_PIXEL_STRATEGIES)
SG_WINDOW_CHOICES = [7, 9, 11, 13, 15]
SG_POLYORDER_CHOICES = [2]
POSITION_DILATION_RADIUS_CHOICES = [2, 3, 4, 5]

USE_WAVELENGTH_WINDOW = False
WINDOW_MIN_NM = 1225.0
WINDOW_MAX_NM = 1675.0

RUN_OPTUNA = True
LOAD_EXISTING_STUDY = True
N_TRIALS_PER_FAMILY = {
    "object_matrix": 200,
    "pixel_matrix": 300,
}
OBJECTIVE_SEEDS = [0, 1, 2]
MAX_FN_RATE_FOR_THRESHOLD = 0.00
MAX_FP_RATE_FOR_THRESHOLD = 0.50
RANDOM_STATE = expcfg.RANDOM_STATE
REPLACE_BALANCED_PIXELS = expcfg.REPLACE_BALANCED_PIXELS

track_specs_df = pd.DataFrame([
    {"selection_track": track, **spec}
    for track, spec in expcfg.SIMCA_SELECTION_TRACK_SPECS.items()
])

display(track_specs_df)
print("DB_H5_PATH:", DB_H5_PATH)
print("PCA_SELECTED_PREPROCESSINGS_PATH:", PCA_SELECTED_PREPROCESSINGS_PATH)
print("GRID_MODEL_CANDIDATES_PATH:", GRID_MODEL_CANDIDATES_PATH)
print("RESULTS_DIR:", RESULTS_DIR)

,selection_track,matrix_family,decision_mode,primary_metric_level,secondary_metric_level
0,object_matrix_2way,object_matrix,2way,object,pixel
1,object_matrix_3way,object_matrix,3way,object,pixel
2,pixel_matrix_2way,pixel_matrix,2way,pixel,object
3,pixel_matrix_3way,pixel_matrix,3way,pixel,object


DB_H5_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\HSI Data\processed\nir_uco_database.h5
PCA_SELECTED_PREPROCESSINGS_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03_pca_non_noisy_all\pca_selected_preprocessings.parquet
GRID_MODEL_CANDIDATES_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_grid_search_non_noisy_all\grid_model_candidates.parquet
RESULTS_DIR: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_optuna_search_non_noisy_all


## Load inputs

In [3]:
if not DB_H5_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_H5_PATH}. Run notebook 00 first.")
if not PCA_SELECTED_PREPROCESSINGS_PATH.exists():
    raise FileNotFoundError(
        f"PCA shortlist not found: {PCA_SELECTED_PREPROCESSINGS_PATH}. Run notebook 03 first."
    )
if not GRID_MODEL_CANDIDATES_PATH.exists():
    raise FileNotFoundError(
        f"04A grid candidates not found: {GRID_MODEL_CANDIDATES_PATH}. Run notebook 04A first."
    )

object_db, image_db = load_nir_uco_h5(
    DB_H5_PATH,
    reconstruct_heavy_object_arrays=True,
)

if USE_WAVELENGTH_WINDOW:
    object_db, image_db, wavelengths, wavelength_info = select_wavelength_range_from_database(
        object_db=object_db,
        image_db=image_db,
        min_nm=WINDOW_MIN_NM,
        max_nm=WINDOW_MAX_NM,
    )
    wavelength_selection_df = wavelength_selection_summary(wavelength_info)
else:
    first_obj = next(iter(object_db.values()))
    wavelengths = first_obj.get("wavelengths")
    wavelengths = np.asarray(wavelengths, dtype=float) if wavelengths is not None else None
    wavelength_selection_df = pd.DataFrame()

if wavelengths is None:
    raise RuntimeError("No wavelength axis found in object_db.")

pca_selected_preprocessings_df = load_parquet(PCA_SELECTED_PREPROCESSINGS_PATH)
grid_model_candidates_df = load_parquet(GRID_MODEL_CANDIDATES_PATH)
preprocessing_configs_by_family = build_pca_preprocessing_configs_by_matrix_family(
    pca_selected_preprocessings_df
)

preprocessing_scope_df = pd.DataFrame([
    {
        "matrix_family": family,
        "preprocessing": name,
        "preprocessing_steps": "+".join(steps),
    }
    for family, configs in preprocessing_configs_by_family.items()
    for name, steps in configs.items()
])

wavelength_config_df = pd.DataFrame([{
    "wavelength_mode": WAVELENGTH_MODE,
    "use_wavelength_window": bool(USE_WAVELENGTH_WINDOW),
    "results_tag": RESULTS_TAG,
    "window_min_nm": WINDOW_MIN_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "window_max_nm": WINDOW_MAX_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "n_bands": int(len(wavelengths)),
    "min_wavelength_nm": float(np.min(wavelengths)),
    "max_wavelength_nm": float(np.max(wavelengths)),
}])

save_parquet(compact_simca_table_for_path(wavelength_config_df, WAVELENGTH_CONFIG_PATH), WAVELENGTH_CONFIG_PATH)
save_parquet(compact_simca_table_for_path(preprocessing_scope_df, PREPROCESSING_SCOPE_PATH), PREPROCESSING_SCOPE_PATH)

print("Number of images:", len(image_db))
print("Number of objects:", len(object_db))
print("04A grid candidates:", grid_model_candidates_df.shape)
display(wavelength_config_df)
display(preprocessing_scope_df.sort_values(["matrix_family", "preprocessing"]))

Number of images: 48
Number of objects: 1262
04A grid candidates: (3240, 44)


,wavelength_mode,use_wavelength_window,results_tag,window_min_nm,window_max_nm,n_bands,min_wavelength_nm,max_wavelength_nm
0,non_noisy_all,False,non_noisy_all,NaN,NaN,63,960.735294,1702.0


,matrix_family,preprocessing,preprocessing_steps
0,object_matrix,absorbance_sg_d1,absorbance+sg_d1
1,object_matrix,absorbance_sg_d2,absorbance+sg_d2
4,object_matrix,absorbance_snv_sg_d1,absorbance+snv+sg_d1
2,object_matrix,absorbance_snv_sg_d2,absorbance+snv+sg_d2
3,object_matrix,snv_sg_d2,snv+sg_d2
6,pixel_matrix,absorbance_snv_sg_smooth,absorbance+snv+sg_smooth
9,pixel_matrix,raw,raw
8,pixel_matrix,sg_smooth,sg_smooth
7,pixel_matrix,snv,snv
5,pixel_matrix,snv_sg_smooth,snv+sg_smooth


## Run Optuna

Optuna optimizes validation 2-way object-level metrics. The generated model candidates are later refit and evaluated for both 2-way and 3-way decision layers.

In [4]:
optuna_trials_parts = []
studies = {}

for matrix_family, space in OPTUNA_MATRIX_FAMILY_SPACES.items():
    print("=" * 80)
    print(f"Optuna study for matrix_family={matrix_family}")
    print("=" * 80)

    storage_path = Path(space["storage_path"])
    storage_path.parent.mkdir(parents=True, exist_ok=True)

    objective = make_optuna_binary_pareto_objective(
        object_db=object_db,
        image_db=image_db,
        train_filters=TRAIN_FILTERS,
        projection_filters=VALIDATION_FILTERS,
        preprocessing_configs=preprocessing_configs_by_family,
        matrix_methods=space["matrix_methods"],
        rule_variants=RULE_VARIANTS,
        object_thresholds=OBJECT_THRESHOLDS,
        seeds=OBJECTIVE_SEEDS,
        max_fn_rate=MAX_FN_RATE_FOR_THRESHOLD,
        max_fp_rate=MAX_FP_RATE_FOR_THRESHOLD,
        wavelengths=wavelengths,
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
        n_components_choices=N_COMPONENTS_CHOICES,
        alpha_choices=ALPHA_CHOICES,
        m_choices=M_CHOICES,
        balanced_pixel_strategy_choices=BALANCED_PIXEL_STRATEGY_CHOICES,
        sg_window_choices=SG_WINDOW_CHOICES,
        sg_polyorder_choices=SG_POLYORDER_CHOICES,
        position_dilation_radius_choices=POSITION_DILATION_RADIUS_CHOICES,
    )

    sampler = optuna.samplers.TPESampler(
        seed=RANDOM_STATE,
        multivariate=True,
        group=True,
        n_startup_trials=30,
    )
    storage = optuna.storages.RDBStorage(
        url=f"sqlite:///{storage_path.as_posix()}",
        engine_kwargs={"connect_args": {"timeout": 30.0}},
    )

    study = optuna.create_study(
        study_name=space["study_name"],
        directions=["minimize", "minimize", "maximize"],
        sampler=sampler,
        storage=storage,
        load_if_exists=LOAD_EXISTING_STUDY,
    )

    if RUN_OPTUNA:
        study.optimize(
            objective,
            n_trials=int(N_TRIALS_PER_FAMILY[matrix_family]),
            n_jobs=1,
            show_progress_bar=True,
        )

    trials_df = optuna_trials_dataframe(study)
    trials_df["matrix_family_study"] = matrix_family
    trials_df["study_name"] = space["study_name"]
    trials_df["storage_path"] = str(storage_path)
    optuna_trials_parts.append(trials_df)
    studies[matrix_family] = study

    close_optuna_study(study)

optuna_trials_df = (
    pd.concat(optuna_trials_parts, ignore_index=True, sort=False)
    if optuna_trials_parts
    else pd.DataFrame()
)
optuna_completed_trials_df = (
    optuna_trials_df.loc[optuna_trials_df["state"].astype(str).eq("COMPLETE")].copy()
    if len(optuna_trials_df) > 0
    else pd.DataFrame()
)

print("Optuna trials:", optuna_trials_df.shape)
print("Completed trials:", optuna_completed_trials_df.shape)
display(optuna_completed_trials_df.head())

Optuna study for matrix_family=object_matrix


c:\Users\alixg\anaconda3\envs\hsi-nuts\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
c:\Users\alixg\anaconda3\envs\hsi-nuts\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  optuna_warn(
[I 2026-07-21 18:08:08,613] Using an existing study with name '04B_object_matrix_non_noisy_all' instead of creating a new one.


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-07-21 18:08:14,213] Trial 200 finished with values: [1.0, 0.0, 0.5] and parameters: {'matrix_method': 'object_median', 'preprocessing': 'absorbance_snv_sg_d2', 'rule_variant': 'simple_chi2', 'n_components': 3, 'alpha': 0.01, 'sg_window_length': 11, 'sg_polyorder': 2, 'position_dilation_radius': 3}.
[I 2026-07-21 18:08:18,331] Trial 201 finished with values: [1.0, 0.0, 0.5] and parameters: {'matrix_method': 'object_mean', 'preprocessing': 'absorbance_snv_sg_d2', 'rule_variant': 'simple_chi2', 'n_components': 4, 'alpha': 0.01, 'sg_window_length': 7, 'sg_polyorder': 2, 'position_dilation_radius': 4}.
[I 2026-07-21 18:08:21,411] Trial 202 finished with values: [0.6037735849056604, 0.2909090909090909, 0.5526586620926244] and parameters: {'matrix_method': 'object_median', 'preprocessing': 'absorbance_sg_d2', 'rule_variant': 'simple_emp_cv', 'n_components': 4, 'alpha': 0.01, 'sg_window_length': 11, 'sg_polyorder': 2, 'position_dilation_radius': 4}.
[I 2026-07-21 18:08:24,221] Trial 20

c:\Users\alixg\anaconda3\envs\hsi-nuts\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
c:\Users\alixg\anaconda3\envs\hsi-nuts\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  optuna_warn(
[I 2026-07-21 18:17:17,847] Using an existing study with name '04B_pixel_matrix_non_noisy_all' instead of creating a new one.


  0%|          | 0/300 [00:00<?, ?it/s]

[I 2026-07-21 18:17:23,334] Trial 300 finished with values: [0.0, 0.9454545454545454, 0.5272727272727272] and parameters: {'matrix_method': 'balanced_pixels', 'preprocessing': 'sg_smooth', 'rule_variant': 'alternative_empHQ_emp_cv', 'n_components': 7, 'alpha': 0.01, 'm': 60, 'balanced_pixel_strategy': 'random', 'sg_window_length': 7, 'sg_polyorder': 2, 'position_dilation_radius': 5}.
[I 2026-07-21 18:17:27,252] Trial 301 finished with values: [0.0, 1.0, 0.5] and parameters: {'matrix_method': 'balanced_pixels', 'preprocessing': 'snv', 'rule_variant': 'alternative_chi2_emp_cv', 'n_components': 6, 'alpha': 0.01, 'm': 40, 'balanced_pixel_strategy': 'center', 'position_dilation_radius': 2}.
[I 2026-07-21 18:17:32,920] Trial 302 finished with values: [0.0, 1.0, 0.5] and parameters: {'matrix_method': 'balanced_pixels', 'preprocessing': 'snv_sg_smooth', 'rule_variant': 'combined_index_chi2', 'n_components': 10, 'alpha': 0.01, 'm': 80, 'balanced_pixel_strategy': 'center', 'sg_window_length': 15

,number,state,value_0,value_1,value_2,objective_fn_rate_max,objective_fp_rate_mean,objective_balanced_accuracy_mean,value,matrix_method,...,fp_rate_std,m,matrix_family,model_family,object_threshold_median,preprocessing_steps,selection_strategy,matrix_family_study,study_name,storage_path
0,273,COMPLETE,0.000000,0.963636,0.518182,0.000000,0.963636,0.518182,NaN,object_median,...,1.110223e-16,40,object_matrix,rule_variant_grid,0.80,absorbance+sg_d2,optuna_binary_pareto,object_matrix,04B_object_matrix_non_noisy_all,C:\Users\alixg\OneDrive - Université Paris-Dau...
1,374,COMPLETE,0.000000,0.963636,0.518182,0.000000,0.963636,0.518182,NaN,object_median,...,1.110223e-16,40,object_matrix,rule_variant_grid,0.80,absorbance+sg_d2,optuna_binary_pareto,object_matrix,04B_object_matrix_non_noisy_all,C:\Users\alixg\OneDrive - Université Paris-Dau...
2,377,COMPLETE,0.000000,0.963636,0.518182,0.000000,0.963636,0.518182,NaN,object_median,...,1.110223e-16,40,object_matrix,rule_variant_grid,0.80,absorbance+sg_d2,optuna_binary_pareto,object_matrix,04B_object_matrix_non_noisy_all,C:\Users\alixg\OneDrive - Université Paris-Dau...
3,284,COMPLETE,0.018868,0.890909,0.545111,0.018868,0.890909,0.545111,NaN,object_median,...,1.110223e-16,40,object_matrix,rule_variant_grid,0.75,absorbance+sg_d2,optuna_binary_pareto,object_matrix,04B_object_matrix_non_noisy_all,C:\Users\alixg\OneDrive - Université Paris-Dau...
4,323,COMPLETE,0.018868,0.890909,0.545111,0.018868,0.890909,0.545111,NaN,object_median,...,1.110223e-16,40,object_matrix,rule_variant_grid,0.75,absorbance+sg_d2,optuna_binary_pareto,object_matrix,04B_object_matrix_non_noisy_all,C:\Users\alixg\OneDrive - Université Paris-Dau...


## Candidate tables

`optuna_model_candidates.parquet` contains every completed Optuna candidate. `optuna_new_model_candidates.parquet` removes configurations already generated by 04A.

In [5]:
optuna_model_candidates_df = optuna_trials_to_candidate_configs(
    optuna_completed_trials_df,
    n_per_matrix_family=0,
    n_overall=0,
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
    selection_split="validation_batch_3",
    selection_strategy="04B_optuna_search",
)

if optuna_model_candidates_df.empty:
    raise RuntimeError("No completed Optuna candidate could be converted.")

optuna_model_candidates_df = filter_simca_candidates_by_pca_preprocessing(
    optuna_model_candidates_df,
    pca_selected_preprocessings_df,
    strict=True,
)
optuna_model_candidates_df = deduplicate_simca_candidates(
    optuna_model_candidates_df.assign(candidate_source="04B_optuna_search")
)
optuna_model_candidates_df["model_candidate_id"] = optuna_model_candidates_df["candidate_id"]
validate_simca_candidate_contract(optuna_model_candidates_df)

grid_candidate_ids = set(grid_model_candidates_df["candidate_id"].astype(str))
optuna_new_model_candidates_df = (
    optuna_model_candidates_df
    .loc[~optuna_model_candidates_df["candidate_id"].astype(str).isin(grid_candidate_ids)]
    .copy()
    .reset_index(drop=True)
)

optuna_candidate_metrics_df = optuna_model_candidates_df.copy()
optuna_candidate_metrics_df["decision_mode"] = "2way"
optuna_candidate_metrics_df["evaluation_stage"] = "validation_batch_3_multiseed"
optuna_candidate_metrics_df["metric_level"] = "object"
optuna_candidate_metrics_df = add_selection_track(optuna_candidate_metrics_df)
validate_simca_evaluation_contract(optuna_candidate_metrics_df)

print("All Optuna candidates:", optuna_model_candidates_df.shape)
print("New Optuna candidates vs 04A:", optuna_new_model_candidates_df.shape)
display(optuna_model_candidates_df.head())
display(optuna_new_model_candidates_df.head())

All Optuna candidates: (966, 52)
New Optuna candidates vs 04A: (966, 52)


,selected_config_id,selection_split,selection_strategy,candidate_source,optuna_trial_number,optuna_value,value_0,value_1,value_2,objective_fn_rate_max,...,alpha,object_threshold,sg_window_length,sg_polyorder,position_dilation_radius,candidate_id,candidate_sources,n_candidate_sources,n_duplicate_rows,model_candidate_id
0,optuna_0146,validation_batch_3,04B_optuna_search,04B_optuna_search,146,0.327273,0.0,0.448485,0.775758,0.0,...,0.01,0.75,15,2,4,simca_e67086871534e83d,04B_optuna_search,1,2,simca_e67086871534e83d
1,optuna_0083,validation_batch_3,04B_optuna_search,04B_optuna_search,83,-0.063636,0.0,0.709091,0.645455,0.0,...,0.01,0.75,13,2,5,simca_ca87aaf11a2e0148,04B_optuna_search,1,1,simca_ca87aaf11a2e0148
2,optuna_0090,validation_batch_3,04B_optuna_search,04B_optuna_search,90,-0.063636,0.0,0.709091,0.645455,0.0,...,0.01,0.75,15,2,5,simca_1697754d5f4757b4,04B_optuna_search,1,1,simca_1697754d5f4757b4
3,optuna_0079,validation_batch_3,04B_optuna_search,04B_optuna_search,79,-0.118182,0.0,0.745455,0.627273,0.0,...,0.01,0.75,11,2,5,simca_0a6a3e463243ade9,04B_optuna_search,1,1,simca_0a6a3e463243ade9
4,optuna_0497,validation_batch_3,04B_optuna_search,04B_optuna_search,497,-0.118182,0.0,0.745455,0.627273,0.0,...,0.01,0.75,7,2,4,simca_bd506962dd86a0c8,04B_optuna_search,1,1,simca_bd506962dd86a0c8


,selected_config_id,selection_split,selection_strategy,candidate_source,optuna_trial_number,optuna_value,value_0,value_1,value_2,objective_fn_rate_max,...,alpha,object_threshold,sg_window_length,sg_polyorder,position_dilation_radius,candidate_id,candidate_sources,n_candidate_sources,n_duplicate_rows,model_candidate_id
0,optuna_0146,validation_batch_3,04B_optuna_search,04B_optuna_search,146,0.327273,0.0,0.448485,0.775758,0.0,...,0.01,0.75,15,2,4,simca_e67086871534e83d,04B_optuna_search,1,2,simca_e67086871534e83d
1,optuna_0083,validation_batch_3,04B_optuna_search,04B_optuna_search,83,-0.063636,0.0,0.709091,0.645455,0.0,...,0.01,0.75,13,2,5,simca_ca87aaf11a2e0148,04B_optuna_search,1,1,simca_ca87aaf11a2e0148
2,optuna_0090,validation_batch_3,04B_optuna_search,04B_optuna_search,90,-0.063636,0.0,0.709091,0.645455,0.0,...,0.01,0.75,15,2,5,simca_1697754d5f4757b4,04B_optuna_search,1,1,simca_1697754d5f4757b4
3,optuna_0079,validation_batch_3,04B_optuna_search,04B_optuna_search,79,-0.118182,0.0,0.745455,0.627273,0.0,...,0.01,0.75,11,2,5,simca_0a6a3e463243ade9,04B_optuna_search,1,1,simca_0a6a3e463243ade9
4,optuna_0497,validation_batch_3,04B_optuna_search,04B_optuna_search,497,-0.118182,0.0,0.745455,0.627273,0.0,...,0.01,0.75,7,2,4,simca_bd506962dd86a0c8,04B_optuna_search,1,1,simca_bd506962dd86a0c8


## Diagnostics and save

In [6]:
optuna_diagnostics_df = (
    optuna_candidate_metrics_df
    .groupby(
        [
            "selection_track",
            "matrix_family",
            "matrix_method",
            "preprocessing",
            "rule_variant",
        ],
        dropna=False,
    )
    .agg(
        n_candidates=("candidate_id", "nunique"),
        best_fn_rate=("fn_rate", "min"),
        best_fp_rate=("fp_rate", "min"),
        best_balanced_accuracy=("balanced_accuracy", "max"),
    )
    .reset_index()
    .sort_values(
        ["matrix_family", "best_fn_rate", "best_fp_rate", "best_balanced_accuracy"],
        ascending=[True, True, True, False],
    )
    .reset_index(drop=True)
)

optuna_protocol_df = pd.DataFrame([{
    "notebook": "04B_simca_optuna_search",
    "results_tag": RESULTS_TAG,
    "db_h5_path": str(DB_H5_PATH),
    "pca_selected_preprocessings_path": str(PCA_SELECTED_PREPROCESSINGS_PATH),
    "grid_model_candidates_path": str(GRID_MODEL_CANDIDATES_PATH),
    "train_filters_json": json.dumps(TRAIN_FILTERS, default=str),
    "validation_filters_json": json.dumps(VALIDATION_FILTERS, default=str),
    "matrix_family_spaces_json": json.dumps(
        {
            family: {
                "matrix_methods": space["matrix_methods"],
                "study_name": space["study_name"],
                "storage_path": str(space["storage_path"]),
            }
            for family, space in OPTUNA_MATRIX_FAMILY_SPACES.items()
        },
        default=str,
    ),
    "rule_variants_json": json.dumps(RULE_VARIANTS),
    "n_components_choices_json": json.dumps(N_COMPONENTS_CHOICES),
    "alpha_choices_json": json.dumps(ALPHA_CHOICES),
    "object_thresholds_json": json.dumps(OBJECT_THRESHOLDS),
    "m_choices_json": json.dumps(M_CHOICES),
    "balanced_pixel_strategy_choices_json": json.dumps(BALANCED_PIXEL_STRATEGY_CHOICES),
    "sg_window_choices_json": json.dumps(SG_WINDOW_CHOICES),
    "sg_polyorder_choices_json": json.dumps(SG_POLYORDER_CHOICES),
    "position_dilation_radius_choices_json": json.dumps(POSITION_DILATION_RADIUS_CHOICES),
    "objective_seeds_json": json.dumps(OBJECTIVE_SEEDS),
    "generated_decision_modes_json": json.dumps(["2way"]),
    "downstream_decision_modes_json": json.dumps(list(expcfg.SIMCA_DECISION_MODES)),
    "n_trials": int(len(optuna_trials_df)),
    "n_completed_trials": int(len(optuna_completed_trials_df)),
    "n_optuna_model_candidates": int(len(optuna_model_candidates_df)),
    "n_optuna_new_model_candidates": int(len(optuna_new_model_candidates_df)),
    "optuna_trials_path": str(OPTUNA_TRIALS_PATH),
    "optuna_model_candidates_path": str(OPTUNA_MODEL_CANDIDATES_PATH),
    "optuna_new_model_candidates_path": str(OPTUNA_NEW_MODEL_CANDIDATES_PATH),
}])

save_parquet(compact_simca_table_for_path(optuna_trials_df, OPTUNA_TRIALS_PATH), OPTUNA_TRIALS_PATH)
save_parquet(compact_simca_table_for_path(optuna_completed_trials_df, OPTUNA_COMPLETED_TRIALS_PATH), OPTUNA_COMPLETED_TRIALS_PATH)
save_parquet(compact_simca_table_for_path(optuna_model_candidates_df, OPTUNA_MODEL_CANDIDATES_PATH), OPTUNA_MODEL_CANDIDATES_PATH)
save_parquet(compact_simca_table_for_path(optuna_new_model_candidates_df, OPTUNA_NEW_MODEL_CANDIDATES_PATH), OPTUNA_NEW_MODEL_CANDIDATES_PATH)
save_parquet(compact_simca_table_for_path(optuna_candidate_metrics_df, OPTUNA_CANDIDATE_METRICS_PATH), OPTUNA_CANDIDATE_METRICS_PATH)
save_parquet(compact_simca_table_for_path(optuna_diagnostics_df, OPTUNA_DIAGNOSTICS_PATH), OPTUNA_DIAGNOSTICS_PATH)
save_parquet(compact_simca_table_for_path(optuna_protocol_df, OPTUNA_PROTOCOL_PATH), OPTUNA_PROTOCOL_PATH)

print("Saved:")
for path in [
    OPTUNA_TRIALS_PATH,
    OPTUNA_COMPLETED_TRIALS_PATH,
    OPTUNA_MODEL_CANDIDATES_PATH,
    OPTUNA_NEW_MODEL_CANDIDATES_PATH,
    OPTUNA_CANDIDATE_METRICS_PATH,
    OPTUNA_DIAGNOSTICS_PATH,
    OPTUNA_PROTOCOL_PATH,
]:
    print(" -", path)

display(optuna_diagnostics_df.head(20))
display(list_result_files(RESULTS_DIR).head(20))

Saved:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_optuna_search_non_noisy_all\optuna_trials.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_optuna_search_non_noisy_all\optuna_completed_trials.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_optuna_search_non_noisy_all\optuna_model_candidates.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_optuna_search_non_noisy_all\optuna_new_model_candidates.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_optuna_search_non_noisy_all\optuna_candidate_metrics.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_optuna_search_non_noisy_all\optuna_search_diagnostics.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_optuna_search_non_noisy_all\optuna_search_protocol.parquet


,selection_track,matrix_family,matrix_method,preprocessing,rule_variant,n_candidates,best_fn_rate,best_fp_rate,best_balanced_accuracy
0,object_matrix_2way,object_matrix,object_median,absorbance_sg_d2,data_driven_emp_cv,19,0.000000,0.072727,0.711664
1,object_matrix_2way,object_matrix,object_median,absorbance_sg_d2,simple_emp_cv,32,0.000000,0.090909,0.678731
2,object_matrix_2way,object_matrix,object_median,absorbance_sg_d1,simple_emp_cv,20,0.037736,0.018182,0.713379
3,object_matrix_2way,object_matrix,object_median,absorbance_snv_sg_d2,simple_emp_cv,9,0.037736,0.163636,0.644597
4,object_matrix_2way,object_matrix,object_median,absorbance_snv_sg_d1,alternative_chi2_emp_cv,3,0.056604,0.218182,0.607890
5,object_matrix_2way,object_matrix,object_median,snv_sg_d2,simple_emp_cv,7,0.075472,0.709091,0.524871
6,object_matrix_2way,object_matrix,object_median,snv_sg_d2,alternative_chi2_emp_cv,2,0.094340,0.545455,0.576329
7,object_matrix_2way,object_matrix,object_median,snv_sg_d2,data_driven_emp_cv,7,0.094340,0.672727,0.551458
8,object_matrix_2way,object_matrix,object_median,snv_sg_d2,alternative_empHQ_fixed2,5,0.132075,0.709091,0.551115
9,object_matrix_2way,object_matrix,object_mean,snv_sg_d2,alternative_chi2_emp_cv,2,0.150943,0.109091,0.520926


,file,suffixes,size_mb
0,optuna_pixel_matrix.sqlite3,.sqlite3,1.890625
1,optuna_object_matrix.sqlite3,.sqlite3,1.257812
2,optuna_candidate_metrics.parquet,.parquet,0.070337
3,optuna_new_model_candidates.parquet,.parquet,0.067903
4,optuna_model_candidates.parquet,.parquet,0.067903
5,optuna_completed_trials.parquet,.parquet,0.038926
6,optuna_trials.parquet,.parquet,0.038926
7,optuna_search_protocol.parquet,.parquet,0.025057
8,optuna_search_diagnostics.parquet,.parquet,0.007904
9,wavelength_config.parquet,.parquet,0.003999
